In [ ]:
# Allow importing from src
import sys
sys.path.insert(0, '../src/')

In [ ]:
import open3d as o3d
from pathlib import Path
import numpy as np
from matplotlib import pyplot as plt

# Params for Run All scenarios

In [ ]:
Loading_PointCloud_Drawing = True
Alpha_Shapes_Mesh_Drawing = True
Ball_Pivoting_Mesh_Drawing = True
Poisson_Mesh_Drawing = True

# Loading

In [ ]:
point_cloud = o3d.io.read_point_cloud(Path("demo_tape_raymeshnerf.ply"))

print(point_cloud)

point_cloud.paint_uniform_color([0.5, 0.5, 0.5])
point_cloud.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(max_nn=50, radius=0.1))
point_cloud.orient_normals_consistent_tangent_plane(100)

if Loading_PointCloud_Drawing:
    o3d.visualization.draw_geometries(
        [point_cloud],
        zoom=1,
        front=[0, 0, -1],
        lookat=[0, 0, 0],
        up=[0, 1, 0],
        point_show_normal=True,
    )

# Base mesh generation methods

## Alpha shapes

In [ ]:
alpha = 0.2
print(f"alpha={alpha:.3f}")
mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(point_cloud, alpha)
mesh.compute_vertex_normals()

if Alpha_Shapes_Mesh_Drawing:
    o3d.visualization.draw_geometries([mesh], mesh_show_back_face=False)

In [ ]:
mesh_smp = mesh.filter_smooth_taubin(number_of_iterations=20, lambda_filter=0.5, mu=-0.5)
mesh_smp.compute_vertex_normals()
mesh_smp.normalize_normals()

if Alpha_Shapes_Mesh_Drawing:
    o3d.visualization.draw_geometries([mesh_smp])

In [ ]:
voxel_size = max(mesh_smp.get_max_bound() - mesh_smp.get_min_bound()) / 32
print(f'voxel_size = {voxel_size:e}')
mesh_downs = mesh_smp.simplify_vertex_clustering(
    voxel_size=voxel_size,
    contraction=o3d.geometry.SimplificationContraction.Average)
print(
    f'Simplified mesh has {len(mesh_downs.vertices)} vertices and {len(mesh_downs.triangles)} triangles'
)
o3d.visualization.draw_geometries([mesh_downs])

voxel_size = max(mesh_smp.get_max_bound() - mesh_smp.get_min_bound()) / 16
print(f'voxel_size = {voxel_size:e}')
mesh_downs = mesh_smp.simplify_vertex_clustering(
    voxel_size=voxel_size,
    contraction=o3d.geometry.SimplificationContraction.Average)
print(
    f'Simplified mesh has {len(mesh_downs.vertices)} vertices and {len(mesh_downs.triangles)} triangles'
)
o3d.visualization.draw_geometries([mesh_downs])

# Ball pivoting

Couldn't find parameters that result in a full mesh without holes

In [ ]:
radii = [0.08, 0.16]
mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(point_cloud, o3d.utility.DoubleVector(radii))

if Ball_Pivoting_Mesh_Drawing:
    o3d.visualization.draw_geometries([mesh])

# Poisson surface reconstruction

In [ ]:
with o3d.utility.VerbosityContextManager(o3d.utility.VerbosityLevel.Debug) as cm:
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(point_cloud, depth=6)

mesh.compute_vertex_normals()

if Poisson_Mesh_Drawing:
    o3d.visualization.draw_geometries([mesh])

In [ ]:
densities = np.asarray(densities)
density_colors = plt.get_cmap('plasma')(
    (densities - densities.min()) / (densities.max() - densities.min()))
density_colors = density_colors[:, :3]
density_mesh = o3d.geometry.TriangleMesh()
density_mesh.vertices = mesh.vertices
density_mesh.triangles = mesh.triangles
density_mesh.triangle_normals = mesh.triangle_normals
density_mesh.vertex_colors = o3d.utility.Vector3dVector(density_colors)
o3d.visualization.draw_geometries([density_mesh])